# 岩石强度 Weibull 参数 Python 复现示例

这个 Notebook 演示如何使用仓库中的 `rock_weibull_model.py`：

- 构造峰值参数 `PeakState`
- 求解给定 `lambda` 的 Weibull 参数 `m` 和 `F0`
- 读取论文中的 Table 3 / Table 4 数据
- 生成并绘制应力-应变曲线


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

project_root = Path.cwd().resolve()
if not (project_root / 'rock_weibull_model.py').exists():
    project_root = project_root.parent

sys.path.append(str(project_root))

from rock_weibull_model import (
    PeakState,
    solve_weibull_parameters,
    stress_strain_curve,
    TABLE3_RELATIVE_DIMENSION,
    TABLE4_SINGULARITY,
    WeibullParameters,
)

project_root


In [ ]:
peak = PeakState(
    E=15000.0,
    nu=0.25,
    sigma3=0.0,
    sigma1c=35.0,
    epsilon1c=0.004,
    phi_deg=35.0,
)
peak


In [ ]:
for lambda_value in [0.9293, 0.9906, 0.6578]:
    params = solve_weibull_parameters(peak, lambda_value)
    print(f'lambda={lambda_value:.4f} -> m={params.m:.4f}, F0={params.F0:.4f} MPa')


In [ ]:
table3 = pd.DataFrame([r.__dict__ for r in TABLE3_RELATIVE_DIMENSION])
table4 = pd.DataFrame([r.__dict__ for r in TABLE4_SINGULARITY])

display(table3)
display(table4)


In [ ]:
plt.figure(figsize=(7, 4))
for row in TABLE3_RELATIVE_DIMENSION:
    params = WeibullParameters(m=row.m, F0=row.F0_MPa)
    curve = stress_strain_curve(peak, row.lambda_value, params)
    xs = [eps for eps, _ in curve]
    ys = [sig for _, sig in curve]
    plt.plot(xs, ys, label=f'spacing={row.spacing_cm} cm')

plt.xlabel('Axial strain')
plt.ylabel('Axial stress (MPa)')
plt.title('Stress-strain family from Table 3')
plt.grid(True, linestyle='--', alpha=0.4)
plt.legend()
plt.show()


你可以把上面的 `peak` 参数替换为自己的试验参数，然后重新运行求解与绘图。
